# Project: Next Token Predictor Transformer

(Onni Kivinen, Tony Karlin, Joni Heikkilä, Jarkko Kärki)

## 1. Introduction

The main goal of this project was to follow the methods and principles learned during this course. That includes constructing a higher level neural network that utilizes some of the best practices in modern machine learning, such as
- A well defined train/validation/test split.
- Preprocessing the data to better fit our model's needs.
- Converting the data into tokens (tokenization) and in our case using subword tokenization.
    - Subword tokenization splits words into smaller, frequently occurring units (subwords). This reduces the out‑of‑vocabulary problem and helps the model handle rare words by representing them as combinations of common pieces of simpler words.
- A well defined system for tuning hyperparameters. (Clear definitions for each of the variables that represent a tunable hyperparameter)
- A carefully considered resource-aware approach for building our transformer model so that it doesn't use too many resources and stays lightweight during training, while still generalizing considerably well with new unseen data.

The aim with our project was to try and achieve the next token prediction model that could come up with any somewhat reasonable output sequence from a number of input sequences. We used the ``WikiText-2`` / ``WikiText-103`` datasets as the source of our training, validation, and test data. The datasets include a large collection of good and featured Wikipedia articles. That makes the dataset a good benchmark for language modeling, since it contains a diverse topics and writing styles.

Our model followed the **transformer decoder** architecture, which utilizes
- positional encoding and embedding,
- masked multi-head self-attention,
- feedforward,
- layer normalization and
- residual connection components.

The decoder's job was to take the input token sequence and, using masked self-attention, predict the next token at each following position.

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import re
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from keras_hub.layers import TransformerDecoder, TokenAndPositionEmbedding
import sentencepiece as spm
import kagglehub

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Speedrun for RTX 3080 COMMENT THIS IF NEEDED
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

tf.config.experimental.enable_tensor_float_32_execution(True)

In [ ]:
# Can probably be commented out if you have no problems with GPU / CUDA!!!
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))
tf.config.optimizer.set_jit(True)               # SET THIS TO FALSE IF NEEDED
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

2.20.0
[]


## 2. Data

We used kagglehub module for downloading our ``WikiText`` datasets. We loaded the dataset’s pre-defined splits training, validation, and test from the files:

- **wiki.train.tokens**
- **wiki.valid.tokens**
- **wiki.test.tokens**

In [ ]:
DATASET_NAME = "wikitext-103" # change to wikitext-2 for the smaller dataset

path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, DATASET_NAME)

def load_lines(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.readlines()


train_lines = load_lines(wikitext2_path, "wiki.train.tokens")
val_lines   = load_lines(wikitext2_path, "wiki.valid.tokens")
test_lines  = load_lines(wikitext2_path, "wiki.test.tokens")

## 3. Data Cleaning

The raw ``WikiText`` files contain markup-like artifacts (section headers, tokenization markers) and inconsistent whitespace that can hurt both tokenization and training. Before training the tokenizer and the model, we normalize each line with a lightweight cleaning function utilizing regular expressions.

The regular expressions remove
- Remove WikiText section titles: lines like ``== Some Title ==``
- Remove unknown placeholders: delete ``<unk>`` tokens.
- Fix WikiText punctuation markers: convert ``@-@`` → **-** ``@,@`` → **,** ``@.@`` → **.**
- Normalize spacing around punctuation and brackets:
    - remove spaces before ``, . ; : ! ?``
    - remove spaces right after opening brackets ``( [ {``
    - remove spaces right before closing brackets ``) ] }``
    - Collapse repeated whitespace into a single space.

After cleaning the data, we remove short lines (4 tokens or shorter) that often add noise and aren't *that* useful for training next-token models.

In [27]:
def clean_line(text_line):
    text_line = text_line.strip()

    # Drop WikiText section titles; they are metadata, not natural sentences.
    if re.match(r"^=+\s*[^=]+?\s*=+$", text_line):
        return ""

    text_line = text_line.replace("<unk>", "")
    text_line = text_line.replace("@-@", "-")
    text_line = text_line.replace("@,@", ",")
    text_line = text_line.replace("@.@", ".")
    text_line = re.sub(r"\s+([,.;:!?])", r"\1", text_line)
    text_line = re.sub(r"([([{])\s+", r"\1", text_line)
    text_line = re.sub(r"\s+([)\]}])", r"\1", text_line)
    text_line = re.sub(r"\s+", " ", text_line)
    return text_line.strip()


train_lines_clean = [clean_line(text_line) for text_line in train_lines]
val_lines_clean   = [clean_line(text_line) for text_line in val_lines]
test_lines_clean  = [clean_line(text_line) for text_line in test_lines]

train_lines_clean = [line for line in train_lines_clean if len(line.split()) >= 4]
val_lines_clean   = [line for line in val_lines_clean if len(line.split()) >= 4]
test_lines_clean  = [line for line in test_lines_clean if len(line.split()) >= 4]


KeyboardInterrupt: 

## 4. Subword Tokenizer

This section implements Googles ``SentencePiece``, which is a unsupervised text tokenizer with Byte Pair Encoding. Instead of word-level tokenization (which suffers from a large vocabulary and many out-of-vocabulary words), we use subword tokenization. Subword units let the model represent rare or unseen words as combinations of frequent pieces, while keeping the vocabulary size manageable.

The section contains the **vocabulary size** as our models first hyperparameter. Based on the WikiText dataset used, our vocabulary size has been changing between
- ``WikiText-2``: **8000 - 12000**
- ``WikiText-103``: **12000 - 30000**

In [ ]:
with open("wikitext_train.txt", "w", encoding="utf-8") as f:
    for line in train_lines_clean:
        f.write(line + "\n")

In [ ]:
VOCAB_SIZE = 12000

if not os.path.exists("spm_wikitext.model"):
    for f in ["spm_wikitext.model", "spm_wikitext.vocab"]:
        if os.path.exists(f):
            os.remove(f)

    spm.SentencePieceTrainer.train(
        input="wikitext_train.txt",
        model_prefix="spm_wikitext",
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        input_sentence_size=500000,
    )

sp = spm.SentencePieceProcessor()
sp.load("spm_wikitext.model")


True

Here the SentencePiece adds an integer ID for an End of Sequence `<eos>` token that gets appended to every line later.

In [ ]:
eos = sp.eos_id()

train_sequences = sp.encode(train_lines_clean, out_type=int)
val_sequences   = sp.encode(val_lines_clean, out_type=int)
test_sequences  = sp.encode(test_lines_clean, out_type=int)

Here we encode each cleaned text line into subword token IDs that the model can train on. *(Example: [42, 912, 545, 4, 3] where 3 is the ``<eos>`` token)*

And lastly we flatten the list of sequences into one long token stream.

In [ ]:
train_sequences = [seq + [eos] for seq in train_sequences]
val_sequences   = [seq + [eos] for seq in val_sequences]
test_sequences  = [seq + [eos] for seq in test_sequences]

train_sequence = [tok for seq in train_sequences for tok in seq]
val_sequence   = [tok for seq in val_sequences for tok in seq]
test_sequence  = [tok for seq in test_sequences for tok in seq]

vocab_size = sp.get_piece_size()

print("Train tokens:", len(train_sequence))
print("Val tokens:", len(val_sequence))
print("Test tokens:", len(test_sequence))
print("Vocab size:", vocab_size)

Train tokens: 125741485
Val tokens: 260008
Test tokens: 294613
Vocab size: 12000


## 5. Dataset and Hyperparameters

In [ ]:
# Hyperparameters
BATCH_SIZE = 96
TRAIN_STRIDE = 96
VAL_STRIDE = 96
TEST_STRIDE = 96

SEQ_LEN = 96

MAX_TRAIN_BATCHES = 3500  # Set to an integer, e.g. 4000, for a quicker training run

train_tokens = np.array(train_sequence, dtype=np.int32)
val_tokens   = np.array(val_sequence, dtype=np.int32)
test_tokens  = np.array(test_sequence, dtype=np.int32)

# HUGE speed improvement
# 20M tokens is more than enough for this model size
train_tokens = train_tokens[:20_000_000]


def make_dataset(tokens, seq_len, batch_size, shuffle=False, stride=96):
    tokens = np.asarray(tokens, dtype=np.int32)

    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=tokens,
        targets=None,
        sequence_length=seq_len + 1,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=False,
    )

    ds = ds.map(lambda window: (window[:, :-1], window[:, 1:]), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(
    train_tokens,
    SEQ_LEN,
    BATCH_SIZE,
    shuffle=True,
    stride=TRAIN_STRIDE,
)

if MAX_TRAIN_BATCHES is not None:
    train_ds = train_ds.take(MAX_TRAIN_BATCHES)

val_ds = make_dataset(
    val_tokens,
    SEQ_LEN,
    BATCH_SIZE,
    stride=VAL_STRIDE,
)

test_ds = make_dataset(
    test_tokens,
    SEQ_LEN,
    BATCH_SIZE,
    stride=TEST_STRIDE,
)


2026-05-07 17:46:02.385007: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 17:46:02.388620: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 17:46:02.391677: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [ ]:
print("characters:", len(train_lines_clean))
print("tokens:", len(train_tokens))
print("steps:", tf.data.experimental.cardinality(train_ds).numpy())

characters: 1110788
tokens: 20000000
steps: 2171


## 6. Model

In [ ]:
# Hyperparameters
EMBEDDING_DIM = 192
EMBED_MULTIPLIER = 4
NUM_HEADS = 6
DECODER_AMOUNT = 3
DROPOUT_RATE = 0.1

inputs = keras.Input(shape=(None,), dtype="int64", name="inputs")
x = TokenAndPositionEmbedding(
    vocabulary_size=vocab_size,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBEDDING_DIM,
    mask_zero=False,
)(inputs)

for _ in range(DECODER_AMOUNT):
    x = TransformerDecoder(
        intermediate_dim=EMBEDDING_DIM * EMBED_MULTIPLIER,
        num_heads=NUM_HEADS,
        dropout=DROPOUT_RATE,
    )(x)

x = layers.Dropout(DROPOUT_RATE)(x)
# outputs = layers.Dense(vocab_size)(x)  # SparseCategoricalCrossentropy uses logits directly.
outputs = layers.Dense( # changed to this
    vocab_size,
    dtype="float32"
)(x)

transformer_model = keras.Model(inputs, outputs, name="decoder_only_transformer")


In [ ]:
transformer_model.summary()

Model: "decoder_only_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 192)      │     2,322,432 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder             │ (None, None, 192)      │       444,864 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_1           │ (None, None, 192)      │       444,864 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_2           │ (None, None, 192)      │       444,864 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 192)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 12000)    │     2,316,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,973,024 (22.79 MB)

 Trainable params: 5,973,024 (22.79 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Training

In [ ]:
USE_EARLY_STOPPING = True


def make_callbacks(ckpt_path, use_early_stopping=USE_EARLY_STOPPING):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=4,
                min_delta=0.001,
                start_from_epoch=4,
                restore_best_weights=True,
                verbose=1,
            )
        )

    return callbacks


In [ ]:
class Perplexity(keras.metrics.Metric):
    def __init__(self, name="perplexity", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_tracker = keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )

        self.loss_tracker.update_state(loss, sample_weight=sample_weight)

    def result(self):
        return tf.exp(self.loss_tracker.result())

    def reset_state(self):
        self.loss_tracker.reset_state()

In [ ]:
EPOCHS = 24

# The saved model
# if os.path.exists("best_model.keras"):
#     transformer_model = keras.models.load_model(
#     "best_model.keras",
#     compile=False
# )
    
# transformer_model.compile(
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=[
#         "accuracy",
#         keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
#         Perplexity(),
#     ],
# )

# THE ACTUAL TRAINABLE MODEL DO NOT REMOVE!!

transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
        clipnorm=1.0,
    ),
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
        Perplexity(),
    ],
    jit_compile=True,   # set to false if needed
)

transformer_history = transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=make_callbacks("best_model_new.keras"),
)


Epoch 1/24


I0000 00:00:1778165166.224949   33211 service.cc:145] XLA service 0x7fea40008960 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778165166.224972   33211 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3080, Compute Capability 8.6
2026-05-07 17:46:06.272851: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1778165166.294050   33211 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-07 17:46:10.324399: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1778165170.414563   33211 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778165170.414609   33211 assert_op.cc:3

1951/2171 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.1201 - loss: 6.5692 - perplexity: 952.5659 - top_5_accuracy: 0.2454

W0000 00:00:1778165222.003949   33212 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778165222.003966   33212 assert_op.cc:38] Ignoring Assert operator SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778165227.841356   34084 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_20', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1778165228.101892   34088 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_22', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778165228.265846   34085 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_19', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1778165228.311374   34096 asm_compiler.cc:369] ptxas warning : Registers are spille

2171/2171 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1234 - loss: 6.5015 - perplexity: 893.0782 - top_5_accuracy: 0.2513

W0000 00:00:1778165239.965670   33208 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778165239.965734   33208 assert_op.cc:38] Ignoring Assert operator SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778165240.568833   34432 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778165240.988503   34421 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 88 bytes spill stores, 88 bytes spill loads

W0000 00:00:1778165242.432695   33210 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778165242.432716   33210 assert_op.cc:38] Ignoring Assert operator SparseSoftmaxCr


Epoch 1: val_loss improved from None to 5.13232, saving model to best_model_new.keras

Epoch 1: finished saving model to best_model_new.keras
2171/2171 ━━━━━━━━━━━━━━━━━━━━ 81s 28ms/step - accuracy: 0.1546 - loss: 5.8690 - perplexity: 353.9050 - top_5_accuracy: 0.3064 - val_accuracy: 0.1950 - val_loss: 5.1323 - val_perplexity: 169.4105 - val_top_5_accuracy: 0.3755 - learning_rate: 3.0000e-04
Epoch 2/24
2171/2171 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1935 - loss: 5.1000 - perplexity: 164.1590 - top_5_accuracy: 0.3781
Epoch 2: val_loss improved from 5.13232 to 4.77351, saving model to best_model_new.keras

Epoch 2: finished saving model to best_model_new.keras
2171/2171 ━━━━━━━━━━━━━━━━━━━━ 45s 20ms/step - accuracy: 0.1980 - loss: 5.0316 - perplexity: 153.1774 - top_5_accuracy: 0.3850 - val_accuracy: 0.2195 - val_loss: 4.7735 - val_perplexity: 118.3335 - val_top_5_accuracy: 0.4106 - learning_rate: 3.0000e-04
Epoch 3/24
2171/2171 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.

In [ ]:
for loss in transformer_history.history["val_loss"]:
    print(np.exp(loss))

169.41048844942745
118.3334118763373
100.97408225033227
90.25136867863249
84.04409890934103
79.61367077695478
77.11367618604326
74.53890907735169
72.03798478032009
70.58868008740302
69.14759292175687
67.44521453916407
66.51006988480597
65.65028273890492
64.60846577525889
64.26813899959987
63.93704343552945
62.76685735608142
63.033225474838254
61.88200079641976
61.339855671413524
60.919791498110584
60.38627780142519
60.23782377949768


In [ ]:
test_results = transformer_model.evaluate(test_ds, verbose=1)

for name, value in zip(transformer_model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

30/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3031 - loss: 4.0347 - perplexity: 56.6205 - top_5_accuracy: 0.5072

W0000 00:00:1778166277.725550   33209 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778166277.725566   33209 assert_op.cc:38] Ignoring Assert operator SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778166278.142835   43836 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_3', 24 bytes spill stores, 24 bytes spill loads

I0000 00:00:1778166278.185829   43842 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 88 bytes spill stores, 88 bytes spill loads

I0000 00:00:1778166278.645898   43837 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_18', 108 bytes spill stores, 172 bytes spill loads

I0000 00:00:1778166278.766173   43846 asm_compiler.cc:369] ptxas warning : Registers are 

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.3014 - loss: 4.0855 - perplexity: 59.4742 - top_5_accuracy: 0.5035
loss: 4.0855
compile_metrics: 0.3014


## 8. Sampling

In [ ]:
def sample_probs(logits, temperature=1.0):
    temperature = max(float(temperature), 1e-6)
    logits = np.asarray(logits).astype("float64") / temperature
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)


def top_p_filter(probs, top_p=0.9, min_tokens_to_keep=3):
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumulative = np.cumsum(sorted_probs)

    keep = cumulative <= top_p
    keep[:min_tokens_to_keep] = True
    filtered_idx = sorted_idx[keep]
    filtered_probs = probs[filtered_idx]
    filtered_probs = filtered_probs / filtered_probs.sum()
    return filtered_idx, filtered_probs


In [ ]:
BAD_PIECES = {
    "<pad>", "<unk>", "<s>",
    "<", ">", "▁<", "▁>",
    "__&", "__/",
    "@-@", "▁@-@", " "
}

BAD_IDS = {
    sp.pad_id(),
    sp.unk_id(),
    sp.bos_id(),
}

for piece in BAD_PIECES:
    piece_id = sp.piece_to_id(piece)
    if piece_id != sp.unk_id():
        BAD_IDS.add(piece_id)


def encode_prompt(text):
    seq = sp.encode(text, out_type=int)
    seq = seq[-SEQ_LEN:]
    return np.array([seq], dtype=np.int32)


def repetition_adjusted_logits(logits, generated_ids, penalty=1.2, recent_window=80, hard_window=6):
    logits = np.array(logits, dtype="float64", copy=True)

    for bad_id in BAD_IDS:
        if 0 <= bad_id < len(logits):
            logits[bad_id] = -1e9

    recent_ids = generated_ids[-recent_window:]
    hard_block_ids = set(generated_ids[-hard_window:])

    for token_id in set(recent_ids):
        if 0 <= token_id < len(logits):
            if token_id in hard_block_ids:
                logits[token_id] = -1e9
            elif logits[token_id] > 0:
                logits[token_id] /= penalty
            else:
                logits[token_id] *= penalty

    return logits


def predict_top_n(text, model, top_n=5, temperature=0.8, filter_bad=True):
    seq = encode_prompt(text)
    logits = model.predict(seq, verbose=0)[0, -1]

    if filter_bad:
        logits = repetition_adjusted_logits(logits, sp.encode(text, out_type=int), penalty=1.05)

    probs = sample_probs(logits, temperature)
    sorted_indices = np.argsort(probs)[::-1]

    results = []
    for token_id in sorted_indices:
        piece = sp.id_to_piece(int(token_id))
        decoded = sp.decode([int(token_id)])
        if not decoded.strip() and piece != "▁":
            continue

        results.append((decoded, round(float(probs[token_id]) * 100, 2)))
        if len(results) == top_n:
            break

    return results


def sample_next_token_id(generated_ids, model, temperature=0.8, top_k=50, top_p=0.9):
    seq = np.array([generated_ids[-SEQ_LEN:]], dtype=np.int32)
    logits = model.predict(seq, verbose=0)[0, -1]
    logits = repetition_adjusted_logits(logits, generated_ids)

    probs = sample_probs(logits, temperature)
    top_indices = np.argsort(probs)[::-1][:top_k]
    top_probs = probs[top_indices]
    top_probs = top_probs / top_probs.sum()

    if top_p is not None:
        kept_local_idx, kept_probs = top_p_filter(top_probs, top_p=top_p)
        top_indices = top_indices[kept_local_idx]
        top_probs = kept_probs

    return int(np.random.choice(top_indices, p=top_probs))


def generate_text(seed_text, model, num_tokens=60, temperature=0.75, top_k=60, top_p=0.9):
    generated_ids = sp.encode(seed_text, out_type=int)

    for _ in range(num_tokens):
        next_id = sample_next_token_id(
            generated_ids,
            model,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        if next_id == sp.eos_id():
            break

        generated_ids.append(next_id)

    text = sp.decode(generated_ids)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
# Tuneable settings for making predictions
NUMBER_OF_TOKENS = 40
SETTINGS = [
    ("safe", 0.35, 15, 0.8),
    ("balanced", 0.6, 40, 0.9),
    ("creative", 0.8, 80, 0.95),
]

SEEDS = [
    "The film received generally",
    "The album was released in",
    "The war ended in",
    "The character is described as",
    "in her early years, she was",
    "the city is located in",
    "the population of the town was",
    "according to the report",
]

print("\n=== Transformer: Top-5 seuraavaa sanaa ===")
for i in range(len(SEEDS)):
    print("------------------")
    for piece, pct in predict_top_n(SEEDS[i], transformer_model, top_n=5):
        print(f"  {piece:20s} {pct:.2f}%")

print("\n=== Transformer generoitu teksti ===")
for name, temp, top_k, top_p in SETTINGS:
    print("------------------")
    for seed in SEEDS:
        print()
        print(generate_text(
            seed,
            transformer_model,
            num_tokens=NUMBER_OF_TOKENS,
            temperature=temp,
            top_k=top_k,
            top_p=top_p,
        ))



=== Transformer: Top-5 seuraavaa sanaa ===
------------------
  positive             66.54%
  mixed                21.74%
  favorable            9.83%
  negative             1.47%
  well                 0.10%
------------------
  the                  40.54%
  Japan                10.28%
  North                4.37%
  October              2.49%
  November             2.42%
------------------
  the                  33.16%
  a                    32.43%
  an                   4.10%
  June                 1.30%
  September            1.10%
------------------
  "                    53.75%
  a                    20.32%
  being                8.65%
  the                  4.76%
  having               4.62%
------------------
  a                    17.14%
  the                  6.59%
  not                  4.37%
  also                 4.02%
  able                 2.60%
------------------
  a                    27.50%
  downtown             5.01%
  an                   3.26%
  its               